# Getting started with QNTT.jl

`QNTT.jl` drives [`qntt`](https://github.com/paulcazeaux/QNTensorTrains-rs) — quantum-number-preserving
block-sparse tensor trains for fermionic many-body systems, written in Rust — from Julia.

This notebook covers installation, a first ground-state calculation, and the handful of
conventions worth knowing before anything else. The other notebooks build on it:

- **02 — Ground states**: H₄, LiH, H₂O and N₂ with ALS and MALS.
- **03 — Linear response**: excitation energies from the Riemannian Hessian.

## Setup

The package calls into a shared library that you build once from the Rust workspace:

```sh
cargo build --release -p qntt-capi
```

Then, from the repository root:

```julia
using Pkg
Pkg.activate("notebooks")
Pkg.develop(path = "julia")
Pkg.instantiate()
```

In [1]:
# Use the notebook environment (see notebooks/Project.toml) regardless of which kernel started us.
using Pkg
Pkg.activate(@__DIR__)

  Activating 

project at `~/Dropbox/Workplace/RustPackages/qntt/notebooks`


In [2]:
using QNTT, LinearAlgebra, Printf

QNTT.init()          # size the library's thread pool (0 / no argument = let it choose)
@printf("qntt %s, %d threads\n", QNTT.version(), QNTT.nthreads())

qntt 0.1.0, 18 threads


## Reading integrals

An FCIDUMP file carries the one- and two-electron integrals plus a header giving the orbital
count, electron count and `2·Sz`. The core (nuclear-repulsion) energy comes along with it: the
solvers all return the **electronic** energy, so a total energy is always `λ + e0`.

In [3]:
data = QNTT.read_fcidump(joinpath(@__DIR__, "..", "testdata", "h4.FCIDUMP"))

@printf("norb   = %d\n", QNTT.norb(data))
@printf("nelec  = %d\n", QNTT.nelec(data))
@printf("2Sz    = %d\n", QNTT.ms2(data))
@printf("E_core = %.10f Ha\n", QNTT.e0(data))

norb   = 4
nelec  = 4
2Sz    = 0
E_core = 2.2931012473 Ha


## The symmetry descriptor

A calculation lives in a *space*: the orbital count together with the conserved quantum numbers.
`QnSpace(data)` reads the target sector straight off the FCIDUMP header.

The abelian layer conserves the spin-up and spin-down electron counts separately, exactly — they
index the block structure of every tensor core, so they cannot drift the way a penalty term or a
projection would allow.

In [4]:
space = QNTT.QnSpace(data)

QnSpace(d=4, Nup=2, Ndn=2)

Bond dimensions are not a single number here: **every sector at every bond carries its own rank**.
`maxranks` gives the largest admissible rank at each — beyond that the manifold is the full space
and the tensor train is exact.

`uniform_ranks(space, r)` asks for rank `r` everywhere, capped where the symmetry allows less.
That is the usual way to pick a starting manifold.

In [5]:
r = QNTT.uniform_ranks(space, 4)

@printf("bond   maximal ranks (per sector)   requested\n")
for b in 0:space.d
    @printf("  %d    %-28s %s\n", b, string(QNTT.maxranks(space)[b+1]), string(r[b+1]))
end

bond   maximal ranks (per sector)   requested
  0    [1 0 0; 0 0 0; 0 0 0]        [1 0 0; 0 0 0; 0 0 0]


  1    [1 1 0; 1 1 0; 0 0 0]        [1 1 0; 1 1 0; 0 0 0]
  2    [1 2 1; 2 4 2; 1 2 1]        [1 2 1; 2 4 2; 1 2 1]
  3    [0 0 0; 0 1 1; 0 1 1]        [0 0 0; 0 1 1; 0 1 1]
  4    [0 0 0; 0 0 0; 0 0 1]        [0 0 0; 0 0 0; 0 0 1]


## Compiling the Hamiltonian

The second-quantized operator is built exactly: no screening threshold decides the answer.

This is a one-time cost that grows with the orbital count. At scale, time it before assuming a
run is slow for numerical reasons.

In [6]:
t0 = time()
H = QNTT.Hamiltonian(space, data)
@printf("built in %.3f s\n", time() - t0)
@printf("automaton bond dimensions: %s\n", string(QNTT.automaton_bond_dims(H)))
H

built in 0.015 s
automaton bond dimensions: [1, 25, 214, 27, 1]


Hamiltonian{Float64,Abelian}(QnSpace(d=4, Nup=2, Ndn=2))

## A ground state

`mals` is two-site DMRG: it sweeps, solving a local eigenproblem on each pair of neighbouring
sites and re-splitting by SVD, so the ranks adapt as it goes. `bond_dims = [0]` means "no cap" —
every bond keeps whatever rank the `eps`-truncated SVD gives it.

In [ ]:
x0 = QNTT.tt_randn(space, QNTT.uniform_ranks(space, 8); seed = 1234)

λ, x, _ = QNTT.mals(H, x0; bond_dims = [0], eps = 1e-10, max_iter = 30)

@printf("electronic energy  λ = %.10f Ha\n", λ)
@printf("total energy   λ + E_core = %.10f Ha\n", λ + QNTT.e0(data))
@printf("max bond dimension = %d\n", QNTT.max_bond_rank(x))
@printf("residual ‖Hx − λx‖/‖x‖ = %.2e\n", norm(H * x - λ * x) / norm(x))

## The state is an ordinary Julia value

`x` supports the algebra you would expect. Operations that produce a train return a new one; the
solvers never mutate their input.

In [8]:
@printf("norm(x)          = %.10f\n", norm(x))
@printf("dot(x, x)        = %.10f\n", dot(x, x))
@printf("rayleigh(H, x)   = %.10f\n", QNTT.rayleigh_quotient(H, x))
@printf("expectation(H,x) = %.10f   (⟨x,Hx⟩, NOT divided by ⟨x,x⟩)\n", QNTT.expectation(H, x))

y = 2x - x                       # exact TT algebra, unrounded
@printf("‖2x − x − x‖     = %.2e\n", norm(y - x))

z = round(y; eps = 1e-12)        # deterministic SVD rounding
@printf("ranks after rounding: max bond %d\n", QNTT.max_bond_rank(z))

norm(x)          = 1.0000000000
dot(x, x)        = 1.0000000000
rayleigh(H, x)   = -4.4594886960
expectation(H,x) = -4.4594886960   (⟨x,Hx⟩, NOT divided by ⟨x,x⟩)
‖2x − x − x‖     = 0.00e+00
ranks after rounding: max bond 16


### Two energy accessors, deliberately named apart

| call | value |
|---|---|
| `QNTT.expectation(H, x)` | `⟨x, H·x⟩` — **not** normalized |
| `QNTT.rayleigh_quotient(H, x)` | `⟨x, H·x⟩ / ⟨x, x⟩` |

The unnormalized one is what the Rust library calls `RayleighQuotient`. They agree above only
because `mals` returns a normalized state.

## Looking inside a train

Cores are stored block-sparsely: for each site, one dense block per (family, sector) that the
quantum numbers admit. You can read and write those blocks directly.

In [9]:
@printf("site 1 has %d non-empty blocks\n", length(QNTT.blocks(x, 1)))

for ((fam, sec), m) in sort(collect(QNTT.blocks(x, 1)); by = kv -> (string(kv[1][1]), kv[1][2]))
    @printf("  family %-6s sector %-6s  %d×%d\n", fam, sec, size(m, 1), size(m, 2))
end

site 1 has 16 non-empty blocks
  family dn     sector (0, 0)  1×2


  family dn     sector (0, 1)  1×1
  family dn     sector (1, 0)  1×4
  family dn     sector (1, 1)  1×2
  family empty  sector (0, 0)  1×1
  family empty  sector (0, 1)  1×2
  family empty  sector (1, 0)  1×2
  family empty  sector (1, 1)  1×4
  family occ    sector (0, 0)  1×4
  family occ    sector (0, 1)  1×2
  family occ    sector (1, 0)  1×2
  family occ    sector (1, 1)  1×1
  family up     sector (0, 0)  1×2
  family up     sector (0, 1)  1×4
  family up     sector (1, 0)  1×1
  family up     sector (1, 1)  1×2


On a small system the train can also be materialized densely, in the `4^d` Jordan–Wigner basis —
useful as an oracle, useless beyond a dozen orbitals (the call refuses to allocate rather than
exhausting memory).

In [10]:
xd = Array(x)
@printf("dense length 4^%d = %d, norm %.10f (vs %.10f)\n",
        space.d, length(xd), norm(xd), norm(x))

dense length 4^4 = 256, norm 1.0000000000 (vs 1.0000000000)


## Conventions, once

- **Everything is 0-based**: sites `0:d-1`, bonds `0:d`, sector labels `(a, b)` counting electrons
  strictly to the *left* of a bond.
- Integral arrays are the exception: `QNTT.onebody(data)` and `QNTT.twobody(data)` are ordinary
  1-based Julia arrays, `t[i,j]` and `v[i,j,k,l]` in the chemists' `(ij|kl)` convention.
- Solvers return **electronic** energies; add `QNTT.e0(data)`.
- Failed calls throw `QNTTError` carrying the Rust library's own message.

In [11]:
try
    QNTT.mals(H, x0; bond_dims = Int[])
catch e
    println(e)
end

QNTTError(

2, "mals bond_dims must be non-empty (use [0] for uncapped)")
